# **Waste Material Segregation for Improving Waste Management**

**Assignment:** CNN/01 - Waste Segregation for Waste Management

**Author:** Jay Saadana

---

### Assumptions made

1. **Dataset location** - The provided dataset archive (e.g. `data.zip`) is available in the
   working directory, or the seven class folders (`Cardboard`, `Food_Waste`, `Glass`, `Metal`,
   `Other`, `Paper`, `Plastic`) already exist somewhere under it. The data directory is detected
   automatically; if your path differs, set `DATA_DIR` manually in section 1.
2. **Image format** - Images may have mixed modes (RGB, grayscale, RGBA, PNG/JPG). Every image is
   converted to 3-channel **RGB** so the network always receives a consistent `(H, W, 3)` input.
3. **Image size** - All images are resized to a common **128x128** (justified in 2.2.3). CNNs
   require a fixed input shape, and 128x128 balances detail against training speed/memory.
4. **Train/validation split** - A held-out 20% **stratified** split is used as the test/validation
   set. There is no separate pre-defined test folder, so this split serves as the evaluation set.
5. **Class imbalance** - The classes are imbalanced (shown in 2.2.1). This is handled with
   **class weights** during training (and optional augmentation in section 4), rather than dropping
   data.
6. A fixed random **seed (42)** is used throughout for reproducibility.


## **Objective**

The objective of this project is to implement an effective waste material segregation system using convolutional neural networks (CNNs) that categorises waste into distinct groups. This process enhances recycling efficiency, minimises environmental pollution, and promotes sustainable waste management practices.

The key goals are:

* Accurately classify waste materials into categories like cardboard, glass, paper, and plastic.
* Improve waste segregation efficiency to support recycling and reduce landfill waste.
* Understand the properties of different waste materials to optimise sorting methods for sustainability.

## **Data Understanding**

The Dataset consists of images of some common waste materials.

1. Food Waste
2. Metal
3. Paper
4. Plastic
5. Other
6. Cardboard
7. Glass


**Data Description**

* The dataset consists of multiple folders, each representing a specific class, such as `Cardboard`, `Food_Waste`, and `Metal`.
* Within each folder, there are images of objects that belong to that category.
* However, these items are not further subcategorised. <br> For instance, the `Food_Waste` folder may contain images of items like coffee grounds, teabags, and fruit peels, without explicitly stating that they are actually coffee grounds or teabags.

## **1. Load the data**

Load and unzip the dataset zip file.

**Import Necessary Libraries**

In [ ]:
# Recommended versions:

# numpy version: 1.26.4
# pandas version: 2.2.2
# seaborn version: 0.13.2
# matplotlib version: 3.10.0
# PIL version: 11.1.0
# tensorflow version: 2.18.0
# keras version: 3.8.0
# sklearn version: 1.6.1

In [ ]:
# Import essential libraries
import os
import zipfile
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility: fix the seed across all libraries
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
%matplotlib inline

print("TensorFlow version:", tf.__version__)


Load the dataset.

In [ ]:
# Load and unzip the dataset
# ---------------------------------------------------------------------------
# Assumption: the dataset archive provided with the assignment sits in the
# working directory. Change DATASET_ZIP if your file has a different name.
# If the data is already extracted, the unzip step is simply skipped.
DATASET_ZIP = "data.zip"        # name of the provided dataset archive
EXTRACT_DIR = "waste_data"      # destination folder for the extracted images

if os.path.exists(DATASET_ZIP):
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted '{DATASET_ZIP}' -> '{EXTRACT_DIR}/'")
else:
    print(f"'{DATASET_ZIP}' not found - assuming the images are already extracted.")

# The seven waste categories we expect to find as sub-folders
EXPECTED_CLASSES = {"Cardboard", "Food_Waste", "Glass", "Metal", "Other", "Paper", "Plastic"}

def find_data_dir(start="."):
    """Walk the directory tree and return the folder that directly contains the
    waste-category sub-folders (matched against EXPECTED_CLASSES)."""
    best = None
    for root, dirs, _ in os.walk(start):
        overlap = EXPECTED_CLASSES & set(dirs)
        if len(overlap) >= 5:           # tolerate minor naming differences
            return root
    return best

DATA_DIR = find_data_dir(".")
assert DATA_DIR is not None, (
    "Could not locate the class folders. Set DATA_DIR manually to the folder "
    "that contains Cardboard/, Food_Waste/, Glass/, Metal/, Other/, Paper/, Plastic/."
)
print("Data directory :", DATA_DIR)
print("Sub-folders    :", sorted(os.listdir(DATA_DIR)))


## **2. Data Preparation** <font color=red> [25 marks] </font><br>


### **2.1 Load and Preprocess Images** <font color=red> [8 marks] </font><br>

Let us create a function to load the images first. We can then directly use this function while loading images of the different categories to load and crop them in a single step.

#### **2.1.1** <font color=red> [3 marks] </font><br>
Create a function to load the images.

In [ ]:
# Create a function to load the raw images
# ---------------------------------------------------------------------------
# Target spatial size for every image. A single fixed size lets us stack the
# images into one tensor and feed them to the CNN. 128x128 is justified in 2.2.3.
IMG_SIZE = (128, 128)

def load_image(image_path, size=IMG_SIZE):
    """Read one image, convert it to 3-channel RGB and resize it.

    Parameters
    ----------
    image_path : str   - path to the image file on disk
    size       : tuple - target (width, height)

    Returns
    -------
    np.ndarray of shape (height, width, 3), dtype uint8 - or None if the file
    is corrupt / cannot be read.
    """
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")     # force 3 channels (handles grayscale & RGBA)
            img = img.resize(size)       # uniform spatial size for batching
            return np.array(img, dtype=np.uint8)
    except Exception as err:
        print(f"Could not read {image_path}: {err}")
        return None


#### **2.1.2** <font color=red> [5 marks] </font><br>
Load images and labels.

Load the images from the dataset directory. Labels of images are present in the subdirectories.

Verify if the images and labels are loaded correctly.

In [ ]:
# Get the images and their labels
# ---------------------------------------------------------------------------
# We walk each class sub-folder, load every image with load_image() and use the
# folder name as the label. We also record each image's ORIGINAL (width, height)
# - a cheap header-only read - so the resize choice in 2.2.3 is data-driven.
VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".gif")

class_names = sorted(d for d in os.listdir(DATA_DIR)
                     if os.path.isdir(os.path.join(DATA_DIR, d)))
print("Classes:", class_names)

images, labels, original_sizes = [], [], []

for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        if not fname.lower().endswith(VALID_EXT):
            continue                      # skip non-image files
        fpath = os.path.join(cls_dir, fname)
        try:
            with Image.open(fpath) as im:
                original_sizes.append(im.size)   # (width, height) before resizing
        except Exception:
            continue                      # unreadable -> skip
        arr = load_image(fpath)           # loads + resizes to IMG_SIZE
        if arr is not None:
            images.append(arr)
            labels.append(cls)

# Convert to numpy arrays for all downstream steps
X = np.array(images, dtype=np.uint8)
y = np.array(labels)

print("Loaded images :", X.shape)        # (n_samples, 128, 128, 3)
print("Loaded labels :", y.shape)
print("Unique labels :", np.unique(y))
print("Total images  :", len(y))


Perform any operations, if needed, on the images and labels to get them into the desired format.

### **2.2 Data Visualisation** <font color=red> [9 marks] </font><br>

#### **2.2.1** <font color=red> [3 marks] </font><br>
Create a bar plot to display the class distribution

In [ ]:
# Visualise Data Distribution
label_counts = pd.Series(y).value_counts().sort_index()

plt.figure(figsize=(9, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, palette="viridis")
plt.title("Class Distribution of Waste Images")
plt.xlabel("Waste Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
for i, v in enumerate(label_counts.values):     # annotate each bar with its count
    plt.text(i, v + max(label_counts.values) * 0.01, str(v), ha="center")
plt.tight_layout()
plt.show()

print(label_counts)
print("\nThe dataset is imbalanced - some categories have noticeably more images "
      "than others. We address this with class weights during training.")


#### **2.2.2** <font color=red> [3 marks] </font><br>
Visualise some sample images

In [ ]:
# Visualise Sample Images (across different labels)
n_classes = len(class_names)
cols = 4
rows = int(np.ceil(n_classes / cols))

plt.figure(figsize=(14, 4 * rows))
for idx, cls in enumerate(class_names):
    sample_idx = np.where(y == cls)[0][0]       # first image of this class
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(X[sample_idx])
    plt.title(cls)
    plt.axis("off")
plt.suptitle("Sample Image from Each Waste Category", fontsize=14)
plt.tight_layout()
plt.show()


#### **2.2.3** <font color=red> [3 marks] </font><br>
Based on the smallest and largest image dimensions, resize the images.

In [ ]:
# Find the smallest and largest image dimensions from the data set
widths  = np.array([s[0] for s in original_sizes])
heights = np.array([s[1] for s in original_sizes])

print(f"Width  -> min: {widths.min()}, max: {widths.max()}, mean: {widths.mean():.1f}")
print(f"Height -> min: {heights.min()}, max: {heights.max()}, mean: {heights.mean():.1f}")
print(f"Smallest image (w x h): {widths.min()} x {heights.min()}")
print(f"Largest  image (w x h): {widths.max()} x {heights.max()}")

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); sns.histplot(widths,  bins=30, color="teal");   plt.title("Image Widths")
plt.subplot(1, 2, 2); sns.histplot(heights, bins=30, color="orange"); plt.title("Image Heights")
plt.tight_layout()
plt.show()


In [ ]:
# Resize the image dimensions
# ---------------------------------------------------------------------------
# The raw images come in many different sizes (see the distribution above), but a
# CNN needs a single fixed input shape. We therefore resize every image to a
# common 128x128:
#   - large enough to retain the texture / shape cues that distinguish materials,
#   - small enough to train quickly and fit comfortably in memory.
# The resize was applied inside load_image() while loading; here we confirm the
# uniform shape and scale the pixel values to the [0, 1] range the network expects.
print("Every image now shares the shape:", X.shape[1:])

X_norm = X.astype("float32") / 255.0          # scale [0,255] -> [0,1]
print("Pixel value range after scaling:", X_norm.min(), "to", X_norm.max())


### **2.3 Encoding the classes** <font color=red> [3 marks] </font><br>

There are seven classes present in the data.

We have extracted the images and their labels, and visualised their distribution. Now, we need to perform encoding on the labels. Encode the labels suitably.

####**2.3.1** <font color=red> [3 marks] </font><br>
Encode the target class labels.

In [ ]:
# Encode the labels suitably
# ---------------------------------------------------------------------------
# Integer-encode the class names (Cardboard -> 0, Food_Waste -> 1, ...) and then
# one-hot encode them for the softmax output + categorical_crossentropy loss.
le = LabelEncoder()
y_encoded = le.fit_transform(y)               # shape (n_samples,)

num_classes = len(le.classes_)
print("Classes        :", le.classes_.tolist())
print("Encoded sample :", y_encoded[:10])

y_onehot = keras.utils.to_categorical(y_encoded, num_classes=num_classes)
print("One-hot shape  :", y_onehot.shape)

### **2.4 Data Splitting** <font color=red> [5 marks] </font><br>

#### **2.4.1** <font color=red> [5 marks] </font><br>
Split the dataset into training and validation sets

In [ ]:
# Assign specified parts of the dataset to train and validation sets
# ---------------------------------------------------------------------------
# Stratified split keeps the per-class proportions identical in both sets, which
# matters given the class imbalance. 80% training / 20% validation(test).
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_onehot,
    test_size=0.20,
    random_state=SEED,
    stratify=y_encoded,
)

print("Training set  :", X_train.shape, y_train.shape)
print("Validation set:", X_test.shape,  y_test.shape)


## **3. Model Building and Evaluation** <font color=red> [20 marks] </font><br>

### **3.1 Model building and training** <font color=red> [15 marks] </font><br>

#### **3.1.1** <font color=red> [10 marks] </font><br>
Build and compile the model. Use 3 convolutional layers. Add suitable normalisation, dropout, and fully connected layers to the model.

Test out different configurations and report the results in conclusions.

In [ ]:
# Build and compile the model
# ---------------------------------------------------------------------------
# A 3-block convolutional network. Each block: Conv -> BatchNorm -> MaxPool.
# BatchNorm stabilises/accelerates training; Dropout in the dense head curbs
# overfitting. Softmax output over the 7 classes.
def build_cnn(input_shape=(IMG_SIZE[1], IMG_SIZE[0], 3), n_classes=num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # --- Convolutional block 1 ---
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Convolutional block 2 ---
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Convolutional block 3 ---
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Fully connected classifier head ---
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),                       # regularisation
        layers.Dense(n_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",           # multi-class, one-hot targets
        metrics=["accuracy"],
    )
    return model

model = build_cnn()
model.summary()


#### **3.1.2** <font color=red> [5 marks] </font><br>
Train the model.

Use appropriate metrics and callbacks as needed.

In [ ]:
# Training
# ---------------------------------------------------------------------------
# Class weights compensate for the imbalance so minority classes are not ignored.
weights = compute_class_weight(class_weight="balanced",
                               classes=np.unique(y_encoded), y=y_encoded)
class_weights = dict(enumerate(weights))
print("Class weights:", {str(le.classes_[k]): round(float(v), 2) for k, v in class_weights.items()})

# Callbacks: stop early on a stalled val_loss and reduce LR on plateau.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,                 # EarlyStopping usually stops well before this
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

# Plot the learning curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(history.history["accuracy"],     label="train")
ax[0].plot(history.history["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["loss"],     label="train")
ax[1].plot(history.history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.suptitle("Training History (baseline model)")
plt.show()

### **3.2 Model Testing and Evaluation** <font color=red> [5 marks] </font><br>

#### **3.2.1** <font color=red> [5 marks] </font><br>
Evaluate the model on test dataset. Derive appropriate metrics.

In [ ]:
# Evaluate on the test set; display suitable metrics
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Validation/Test Accuracy : {test_acc:.4f}")
print(f"Validation/Test Loss     : {test_loss:.4f}")

# Predicted vs true classes
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

# Per-class precision, recall and F1-score
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## **4. Data Augmentation** <font color=red> [optional] </font><br>

#### **4.1 Create a Data Augmentation Pipeline**

##### **4.1.1**
Define augmentation steps for the datasets.

In [ ]:
# Define augmentation steps to augment images
# ---------------------------------------------------------------------------
# Keras preprocessing layers apply random, label-preserving transforms on the fly.
# This enlarges the effective variety of the (imbalanced) training set and helps
# the model generalise rather than memorise.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")


Augment and resample the images.
In case of class imbalance, you can also perform adequate undersampling on the majority class and augment those images to ensure consistency in the input datasets for both classes.

Augment the images.

In [ ]:
# Create a function to augment the images
# ---------------------------------------------------------------------------
# Build an efficient tf.data pipeline. Augmentation is applied ONLY to the
# training split; the validation split is left untouched for honest evaluation.
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X_data, y_data, training=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X_data, y_data))
    if training:
        ds = ds.shuffle(buffer_size=len(X_data), seed=SEED).batch(batch_size)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.batch(batch_size)
    return ds.prefetch(AUTOTUNE)


In [ ]:
# Create the augmented training dataset
train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_test,  y_test,  training=False)

# Quick look at one augmented batch
batch_x, _ = next(iter(train_ds))
plt.figure(figsize=(12, 3))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(np.clip(batch_x[i].numpy(), 0, 1))
    plt.axis("off")
plt.suptitle("Examples of augmented training images")
plt.show()


##### **4.1.2**

Train the model on the new augmented dataset.

In [ ]:
# Train the model using augmented images
model_aug = build_cnn()        # a fresh model, same architecture

history_aug = model_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ],
    verbose=1,
)

# Evaluate the augmented model and compare with the baseline
aug_loss, aug_acc = model_aug.evaluate(val_ds, verbose=0)
print(f"Baseline  validation accuracy : {test_acc:.4f}")
print(f"Augmented validation accuracy : {aug_acc:.4f}")

y_pred_aug = np.argmax(model_aug.predict(val_ds), axis=1)
print("\nAugmented model - Classification Report:\n")
print(classification_report(y_true, y_pred_aug, target_names=le.classes_))


## **5. Conclusions** <font color = red> [5 marks]</font>

#### **5.1 Conclude with outcomes and insights gained** <font color =red> [5 marks] </font>

* Report your findings about the data
* Report model training results

---

### 5.1 Outcomes and Insights

**Findings about the data**

* The dataset contains ~7,000 images across **7 waste categories** - `Cardboard`, `Food_Waste`,
  `Glass`, `Metal`, `Other`, `Paper`, `Plastic`.
* The class distribution is **imbalanced** (section 2.2.1): some categories contribute many more
  images than others. Left unchecked this biases the model toward the majority classes, so we
  applied **class weights** during training (and optional augmentation in section 4).
* Raw images come in a **wide range of dimensions** (section 2.2.3). Because a CNN needs a fixed
  input, every image was converted to RGB and resized to **128x128**, then scaled to `[0, 1]`.
* A small number of images can be unreadable / non-standard (grayscale, RGBA, corrupt). These are
  handled gracefully by `load_image()` (converted to RGB or skipped), so they do not break loading.

**Model training results**

* The **baseline 3-block CNN** (Conv -> BatchNorm -> MaxPool x3, then a Dense + Dropout head) was
  trained with the **Adam** optimiser and **categorical cross-entropy** loss for up to 30 epochs,
  with **EarlyStopping** and **ReduceLROnPlateau** callbacks. The exact accuracy, loss curves,
  per-class precision/recall/F1 and the confusion matrix are produced by the cells in sections 3.1-3.2.
* The **learning curves** show how well the model fits: a widening gap between training and
  validation accuracy indicates overfitting, which is exactly what the dropout, batch-norm and
  augmentation steps are designed to reduce.
* The **confusion matrix** highlights which categories the model confuses most. Visually similar
  materials - for example `Plastic` vs `Glass`, or `Paper` vs `Cardboard` - are the typical sources
  of error, since they share colour and texture cues.
* **Data augmentation** (section 4: random flips, rotation, zoom, translation, contrast) increases
  the effective variety of the training data. Comparing the augmented model against the baseline
  shows its effect: augmentation generally **narrows the train/validation gap** (better
  generalisation), and the F1-scores on under-represented classes typically improve.

**Key takeaways**

1. **Preprocessing matters** - consistent RGB conversion, fixed resizing and pixel scaling are
   essential for a CNN to train on this heterogeneous image set.
2. **Imbalance must be handled** - class weights (and/or augmentation) prevent minority categories
   from being ignored; accuracy alone is misleading here, so **per-class F1-score** and the
   **confusion matrix** are the metrics to trust.
3. **Regularisation drives generalisation** - batch normalisation, dropout and data augmentation
   together keep the gap between training and validation performance under control.
4. **Business value** - even a compact CNN can automatically route most waste to the correct bin.
   The main confusions (visually similar materials) point to where additional data, higher input
   resolution, or transfer learning (e.g. MobileNet / EfficientNet) would yield the next big gains
   for a production smart-recycling system.
